# Module 07 worksheet: factual task calculations

Run from this notebook directory. Inputs are exact source copies one directory above; outputs go to `../Worksheet-OutputFiles`. Personal reflections and checklist attestations remain pending in the worksheet.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
BASE = Path.cwd().parent
OUT = BASE / 'Worksheet-OutputFiles'
OUT.mkdir(exist_ok=True)
titanic = pd.read_csv(BASE / 'titanic.csv')
poke = pd.read_csv(BASE / 'pokemon.csv')
penguins_raw = pd.read_csv(BASE / 'penguins.csv')


In [2]:
# Task 1: meaning, ordered category, and proportion.
port_labels = {'C': 'Cherbourg', 'Q': 'Queenstown', 'S': 'Southampton'}
titanic['Embarked'] = titanic['Embarked'].astype('category').cat.rename_categories(port_labels)
class_order = pd.CategoricalDtype(['3rd', '2nd', '1st'], ordered=True)
titanic['class'] = titanic['Pclass'].map({1: '1st', 2: '2nd', 3: '3rd'}).astype(class_order)
above_third = titanic['class'] > '3rd'
meaning_summary = pd.DataFrame({'passengers': [len(titanic)], 'above_3rd': [above_third.sum()], 'share': [above_third.mean()]})
print(titanic[['Embarked', 'class']].dtypes)
print(meaning_summary.to_string(index=False))
meaning_summary.to_csv(OUT / 'meaning_summary.csv', index=False)


Embarked    category
class       category
dtype: object
 passengers  above_3rd    share
        891        400 0.448934


In [3]:
# Task 2: overall includes Legendary; it is not a non-Legendary-only group.
def center_spread(series):
    return {'n': series.count(), 'mean': series.mean(), 'median': series.median(),
            'std': series.std(ddof=1), 'q1': series.quantile(.25), 'q3': series.quantile(.75),
            'iqr': series.quantile(.75) - series.quantile(.25)}
attack_summary = pd.DataFrame({'all': center_spread(poke['Attack']),
                              'Legendary': center_spread(poke.loc[poke['Legendary'], 'Attack'])}).T
print(attack_summary.round(6))
attack_summary.to_csv(OUT / 'attack_center_spread.csv', index_label='group')


               n        mean  median        std     q1     q3   iqr
all        800.0   79.001250    75.0  32.457366   55.0  100.0  45.0
Legendary   65.0  116.676923   110.0  30.348037  100.0  131.0  31.0


In [4]:
# Task 3: remove only the two rows with no measurements, not missing sex.
measurements = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
penguins = penguins_raw.dropna(subset=measurements).copy()
def iqr_fences(series, k=1.5):
    q1, q3 = series.quantile([.25, .75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr
reports, flagged = [], []
for group, frame in [('overall', penguins), *list(penguins.groupby('species'))]:
    low, high = iqr_fences(frame['body_mass_g'])
    mask = frame['body_mass_g'].lt(low) | frame['body_mass_g'].gt(high)
    reports.append({'group': group, 'n': len(frame), 'lower_fence': low, 'upper_fence': high, 'outliers': int(mask.sum())})
    selected = frame.loc[mask].copy()
    selected.insert(0, 'source_row', selected.index)
    selected.insert(0, 'flag_scope', group)
    flagged.append(selected)
outlier_report = pd.DataFrame(reports)
flagged_penguins = pd.concat(flagged, ignore_index=True)
print(outlier_report.to_string(index=False))
print('Flagged rows (original zero-based source index included):')
print(flagged_penguins.to_string(index=False))
outlier_report.to_csv(OUT / 'penguin_outlier_summary.csv', index=False)
flagged_penguins.to_csv(OUT / 'penguin_flagged_rows.csv', index=False)


    group   n  lower_fence  upper_fence  outliers
  overall 342      1750.00      6550.00         0
   Adelie 151      2375.00      4975.00         0
Chinstrap  68      2793.75      4643.75         2
   Gentoo 123      3500.00      6700.00         0
Flagged rows (original zero-based source index included):
flag_scope  source_row   species island  bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g    sex
 Chinstrap         189 Chinstrap  Dream            52.0           20.7              210.0       4800.0   MALE
 Chinstrap         190 Chinstrap  Dream            46.9           16.6              192.0       2700.0 FEMALE


In [5]:
# Task 4: continuous-age bins are [0,12], (12,18], (18,60], (60,infinity].
# This preserves fractional ages rather than creating gaps between integer labels.
age_labels = ['child', 'teen', 'adult', 'senior']
titanic['age_group'] = pd.cut(titanic['Age'], bins=[0, 12, 18, 60, np.inf], labels=age_labels, include_lowest=True)
survival_rates = titanic.pivot_table(index='age_group', columns='Sex', values='Survived', aggfunc='mean', observed=True)
survival_sizes = titanic.pivot_table(index='age_group', columns='Sex', aggfunc='size', observed=True)
print('Rates:'); print(survival_rates)
print('Sizes:'); print(survival_sizes)
print('Highest cell:', survival_rates.stack().idxmax(), 'rate:', survival_rates.stack().max())
survival_rates.to_csv(OUT / 'age_sex_survival_rates.csv')
survival_sizes.to_csv(OUT / 'age_sex_group_sizes.csv')


Rates:
Sex          female      male
age_group                    
child      0.593750  0.567568
teen       0.750000  0.088235
adult      0.778947  0.184573
senior     1.000000  0.105263
Sizes:
Sex        female  male
age_group              
child          32    37
teen           36    34
adult         190   363
senior          3    19
Highest cell: ('senior', 'female') rate: 1.0


In [6]:
# Task 5: rank unique pairs by absolute correlation, retaining the sign.
stats = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
pearson = poke[stats].corr(method='pearson')
spearman = poke[stats].corr(method='spearman')
upper = np.triu(np.ones((len(stats), len(stats)), dtype=bool), k=1)
pair_tables = {}
for method, matrix in [('pearson', pearson), ('spearman', spearman)]:
    pairs = matrix.where(upper).stack().dropna().rename('correlation').reset_index()
    pairs.columns = ['stat_1', 'stat_2', 'correlation']
    pairs['absolute_correlation'] = pairs['correlation'].abs()
    pairs = pairs.sort_values('absolute_correlation', ascending=False).reset_index(drop=True)
    pair_tables[method] = pairs
    print(method, 'matrix:'); print(matrix.round(6))
    print(method, 'strongest three:'); print(pairs.head(3).to_string(index=False))
    matrix.to_csv(OUT / f'{method}_matrix.csv')
    pairs.to_csv(OUT / f'{method}_pairs.csv', index=False)
attack_defense_groups = pd.DataFrame([
    {'Legendary': flag, 'n': len(group), 'pearson': group['Attack'].corr(group['Defense']),
     'spearman': group['Attack'].corr(group['Defense'], method='spearman')}
    for flag, group in poke.groupby('Legendary')
])
print('Attack/Defense within Legendary groups:'); print(attack_defense_groups.to_string(index=False))
attack_defense_groups.to_csv(OUT / 'attack_defense_by_legendary.csv', index=False)


pearson matrix:
               HP    Attack   Defense   Sp. Atk   Sp. Def     Speed
HP       1.000000  0.422386  0.239622  0.362380  0.378718  0.175952
Attack   0.422386  1.000000  0.438687  0.396362  0.263990  0.381240
Defense  0.239622  0.438687  1.000000  0.223549  0.510747  0.015227
Sp. Atk  0.362380  0.396362  0.223549  1.000000  0.506121  0.473018
Sp. Def  0.378718  0.263990  0.510747  0.506121  1.000000  0.259133
Speed    0.175952  0.381240  0.015227  0.473018  0.259133  1.000000
pearson strongest three:
 stat_1  stat_2  correlation  absolute_correlation
Defense Sp. Def     0.510747              0.510747
Sp. Atk Sp. Def     0.506121              0.506121
Sp. Atk   Speed     0.473018              0.473018
spearman matrix:
               HP    Attack   Defense   Sp. Atk   Sp. Def     Speed
HP       1.000000  0.565734  0.433402  0.470813  0.493280  0.265920
Attack   0.565734  1.000000  0.514995  0.361616  0.320620  0.373443
Defense  0.433402  0.514995  1.000000  0.314217  0.578816 

Attack/Defense within Legendary groups:
 Legendary   n   pearson  spearman
     False 735  0.437192  0.501235
      True  65 -0.210974 -0.178946


In [7]:
# Independent checks of the quantities used in the worksheet.
assert len(titanic) == 891 and len(poke) == 800 and len(penguins_raw) == 344
assert titanic['Age'].isna().sum() == 177
assert titanic['Embarked'].isna().sum() == 2
assert titanic['class'].cat.ordered
assert titanic['class'].cat.categories.tolist() == ['3rd', '2nd', '1st']
assert above_third.sum() == titanic['Pclass'].isin([1, 2]).sum() == 400
assert np.isclose(above_third.mean(), 400 / 891)
assert set(titanic['Embarked'].cat.categories) == set(port_labels.values())
for key, series in [('all', poke['Attack']), ('Legendary', poke.loc[poke['Legendary'], 'Attack'])]:
    x = series.dropna().to_numpy()
    assert np.isclose(attack_summary.loc[key, 'mean'], np.mean(x))
    assert np.isclose(attack_summary.loc[key, 'median'], np.median(x))
    assert np.isclose(attack_summary.loc[key, 'std'], np.std(x, ddof=1))
    assert np.isclose(attack_summary.loc[key, 'iqr'], np.percentile(x, 75) - np.percentile(x, 25))
assert len(penguins) == 342
assert penguins_raw[measurements].isna().all(axis=1).sum() == 2
assert penguins[measurements].notna().all().all()
for row in outlier_report.itertuples(index=False):
    frame = penguins if row.group == 'overall' else penguins[penguins['species'].eq(row.group)]
    values = frame['body_mass_g'].to_numpy()
    q1, q3 = np.percentile(values, [25, 75]); low, high = q1 - 1.5*(q3-q1), q3 + 1.5*(q3-q1)
    assert np.isclose(row.lower_fence, low) and np.isclose(row.upper_fence, high)
    assert row.outliers == int(((values < low) | (values > high)).sum())
assert survival_sizes.to_numpy().sum() == 714
assert titanic['age_group'].isna().sum() == 177
for age_group in age_labels:
    for sex in ['female', 'male']:
        group = titanic[titanic['age_group'].eq(age_group) & titanic['Sex'].eq(sex)]
        assert survival_sizes.loc[age_group, sex] == len(group)
        assert np.isclose(survival_rates.loc[age_group, sex], group['Survived'].sum()/len(group))
assert np.isclose(np.average(survival_rates.to_numpy().ravel(), weights=survival_sizes.to_numpy().ravel()),
                  titanic.loc[titanic['Age'].notna(), 'Survived'].mean())
assert poke[stats].notna().all().all()
assert np.allclose(pearson.to_numpy(), np.corrcoef(poke[stats].to_numpy().T))
assert np.allclose(spearman.to_numpy(), poke[stats].rank(method='average').corr().to_numpy())
for table in pair_tables.values():
    assert len(table) == 15 and not (table['stat_1'] == table['stat_2']).any()
    assert table['absolute_correlation'].is_monotonic_decreasing
for row in attack_defense_groups.itertuples(index=False):
    group = poke[poke['Legendary'].eq(row.Legendary)]
    assert row.n == len(group)
    assert np.isclose(row.pearson, np.corrcoef(group['Attack'], group['Defense'])[0, 1])
    assert np.isclose(row.spearman, group[['Attack', 'Defense']].rank().corr().iloc[0, 1])
print('All worksheet verification checks passed.')
print('Output files:', sorted(p.name for p in OUT.glob('*.csv')))


All worksheet verification checks passed.
Output files: ['age_sex_group_sizes.csv', 'age_sex_survival_rates.csv', 'attack_center_spread.csv', 'attack_defense_by_legendary.csv', 'meaning_summary.csv', 'pearson_matrix.csv', 'pearson_pairs.csv', 'penguin_flagged_rows.csv', 'penguin_outlier_summary.csv', 'spearman_matrix.csv', 'spearman_pairs.csv']
